# Full Phase 1 / 2 Pipeline 

Can be used to run and visualize best output prompts on representative (median) image. Creates spider plots for axes comparison of given crop.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

# Resolve project root from notebook location
# notebook is under AgVFM/notebooks, so we want to go up a level to get to the project root
PROJECT_ROOT = Path.cwd().parent
if PROJECT_ROOT.name != "AgVFM":
    candidate = PROJECT_ROOT / "AgVFM"
    if candidate.exists():
        PROJECT_ROOT = candidate

VIZ_DIR = PROJECT_ROOT / "experiments" / "scripts" / "visualization"
EXP_DIR = PROJECT_ROOT / "experiments" / "scripts" / "experiments"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(VIZ_DIR))
sys.path.insert(0, str(EXP_DIR))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VIZ_DIR      = {VIZ_DIR}")

In [ ]:
# imports of phase 1 and phase 2 
from phase1_factor_analysis import (
    plot_all_metrics_factor_contributions,
    plot_axis_spider_comparison,
    plot_spider_summary_grid,
)
from phase1_factor_analysis_hf import (
    plot_hf_model_comparison,
    plot_hf_comparison_all_metrics,
    plot_four_model_comparison,
)

# for default axes (edit if desired)
from agvfm.config.experiments import FACTOR_AXES

# import for median visualization
from viz_median_predictions import _build_model, find_median_image, visualize_two_panel

In [ ]:
# =========================
# User configuration
# =========================
RUN_TAG = "notebook_visuals"
IOU_THRESHOLD = 0.5
METRICS = ["map", "f1", "precision", "recall"]

# Main results directory where load_and_run outputs are stored
RESULTS_DIR = PROJECT_ROOT / "experiments" / "results" / "load_and_run" / "SYN-FLOWER"

# Optional image/label directories for qualitative prediction overlays
IMG_DIR = PROJECT_ROOT / "data" / "all_flower_test"
LBL_DIR = PROJECT_ROOT / "data" / "all_flower_test"

OUT_DIR = PROJECT_ROOT / "experiments" / "results" / "notebook_visualizations" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RESULTS_DIR = {RESULTS_DIR}")
print(f"OUT_DIR     = {OUT_DIR}")

## Run Phase 1 + Phase 2 (`load_and_run.py`) for All Models
Set paths and options below. Run this first when results have not been generated yet.

In [ ]:
import subprocess
import shlex

# Interpreted as the crop/object text passed to --run-ph1 and --run-ph2 in load_and_run.py
TARGET_PROMPT = "cowpea flower"

PIPELINE_IMG_DIR = IMG_DIR
PIPELINE_LBL_DIR = LBL_DIR
PIPELINE_RESULTS_DIR = RESULTS_DIR

DEVICE = "cuda"
SAMPLE_SIZE = None  # set to int (e.g. 200) for fast debugging
NO_EMOJI = False

run_script = PROJECT_ROOT / "experiments" / "scripts" / "experiments" / "load_and_run.py"

cmd = [
    sys.executable,
    str(run_script),
    "--run-ph1", TARGET_PROMPT,
    "--run-ph2", TARGET_PROMPT,
    "--model", "all",
    "--img-dir", str(PIPELINE_IMG_DIR),
    "--lbl-dir", str(PIPELINE_LBL_DIR),
    "--results-dir", str(PIPELINE_RESULTS_DIR),
    "--device", DEVICE,
]

if SAMPLE_SIZE is not None:
    cmd += ["--sample-size", str(SAMPLE_SIZE)]
if NO_EMOJI:
    cmd += ["--no-emoji"]

print("Command:")
print(" \n".join(shlex.quote(c) for c in cmd))

# Set RUN_PIPELINE = True to execute
RUN_PIPELINE = False
if RUN_PIPELINE:
    completed = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
    print("Pipeline finished with return code:", completed.returncode)
else:
    print("Pipeline execution is disabled. Set RUN_PIPELINE=True to run.")

## Load Result JSON Files
After running the pipeline cell (or if results already exist), load the Phase 1/2 outputs here.

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print(f"[missing] {path}")
        return None
    with open(path, "r") as f:
        return json.load(f)

MODEL_KEYS = ["yolo_world", "sam3", "grounding_dino", "owlv2"]

ph1 = {k: load_json(RESULTS_DIR / f"ph1_{k}_factor_analysis.json") for k in MODEL_KEYS}
ph2 = {k: load_json(RESULTS_DIR / f"ph2_{k}_combinations.json") for k in MODEL_KEYS}
summary = load_json(RESULTS_DIR / "best_prompt_summary.json")

available_ph1 = {
    (d.get("model", k) if d else k): d
    for k, d in ph1.items()
    if d is not None
}

print("Available Phase 1 models:", list(available_ph1.keys()))
print("Summary JSON loaded:", summary is not None)

## Full Spider Plots (Primary)
Generate full spider/radar visual summaries first, then optional per-axis spider plots.

In [ ]:
if available_ph1:
    plot_spider_summary_grid(
        available_ph1,
        metrics=["map", "f1"],
        iou_threshold=IOU_THRESHOLD,
        save_dir=OUT_DIR,
    )

    for axis in FACTOR_AXES:
        plot_axis_spider_comparison(
            available_ph1,
            axis.name,
            metrics=["map", "f1"],
            iou_threshold=IOU_THRESHOLD,
            save_dir=OUT_DIR,
            figsize=(9, 9),
        )

    print(f"Spider plots saved under: {OUT_DIR}")
else:
    print("No Phase 1 results available; skipping spider plots.")

In [ ]:
# Secondary figures: per-model all-metrics + model comparisons
for k, d in ph1.items():
    if d is None:
        continue
    plot_all_metrics_factor_contributions(
        d,
        iou_threshold=IOU_THRESHOLD,
        save_path=OUT_DIR / f"{k}_factor_contributions_all_metrics.png",
    )

if ph1["grounding_dino"] is not None and ph1["owlv2"] is not None:
    plot_hf_comparison_all_metrics(
        ph1["grounding_dino"],
        ph1["owlv2"],
        iou_threshold=IOU_THRESHOLD,
        save_path=OUT_DIR / "hf_comparison_all_metrics.png",
    )
    for m in METRICS:
        plot_hf_model_comparison(
            ph1["grounding_dino"],
            ph1["owlv2"],
            metric=m,
            iou_threshold=IOU_THRESHOLD,
            save_path=OUT_DIR / f"hf_comparison_{m}.png",
        )

if all(ph1[k] is not None for k in ["yolo_world", "sam3", "grounding_dino", "owlv2"]):
    for m in METRICS:
        plot_four_model_comparison(
            ph1["yolo_world"],
            ph1["sam3"],
            ph1["grounding_dino"],
            ph1["owlv2"],
            metric=m,
            iou_threshold=IOU_THRESHOLD,
            save_path=OUT_DIR / f"four_model_comparison_{m}.png",
        )

print(f"Secondary plots saved under: {OUT_DIR}")

## Prompt Discovery Summary (Phase 1 and Phase 2)

In [ ]:
def best_row(blob, model_key, phase):
    if not blob:
        return None
    bp = blob.get("best_prompt", {})
    iou05 = bp.get("metrics_by_iou", {}).get("iou_0.5", {})
    return {
        "phase": phase,
        "model": model_key,
        "name": bp.get("name"),
        "prompt": bp.get("prompt"),
        "map_05": iou05.get("map", bp.get("map_05")),
        "f1_05": iou05.get("f1", bp.get("f1_05")),
        "best_conf_05": iou05.get("best_conf", bp.get("best_conf_05")),
    }

rows = []
for k in MODEL_KEYS:
    r1 = best_row(ph1.get(k), k, "phase1")
    r2 = best_row(ph2.get(k), k, "phase2")
    if r1:
        rows.append(r1)
    if r2:
        rows.append(r2)

df_best = pd.DataFrame(rows)
df_best

## Optional: Qualitative Median Prediction Overlay
Use best prompt from Phase 2 (or Phase 1 fallback) for one model and produce a two-panel image: baseline vs best prompt.

In [ ]:
from agvfm.data import get_full_test_paths

MODEL_KEY = "sam3"  # one of: yolo_world, sam3, grounding_dino, owlv2
BASELINE_PROMPT = "a flower"
CONF_SWEEP = 0.05
MIN_GT_BOXES = 5

model_blob = ph2.get(MODEL_KEY) or ph1.get(MODEL_KEY)
if not model_blob:
    raise ValueError(f"No results found for model: {MODEL_KEY}")

bp = model_blob.get("best_prompt", {})
best_prompt = bp.get("prompt")
best_conf = bp.get("best_conf_05", 0.05)

if best_prompt is None:
    raise ValueError("best_prompt text not found in selected results JSON.")

class _Args:
    pass

args_obj = _Args()
args_obj.device = "cuda"
args_obj.yolo_weights = None
args_obj.gdino_model_id = "IDEA-Research/grounding-dino-base"
args_obj.gdino_box_threshold = 0.3
args_obj.gdino_text_threshold = 0.25
args_obj.owlv2_model_id = "google/owlv2-base-patch16-ensemble"
args_obj.sam3_model_id = "facebook/sam3"

image_paths = get_full_test_paths(IMG_DIR)
model = _build_model(MODEL_KEY, args_obj)
median_path, scored = find_median_image(
    model=model,
    image_paths=image_paths,
    lbl_dir=str(LBL_DIR),
    prompt=best_prompt,
    best_conf=float(best_conf),
    min_gt_boxes=MIN_GT_BOXES,
    iou_threshold=IOU_THRESHOLD,
    conf_sweep=CONF_SWEEP,
    nonzero_median=False,
)

if median_path is None:
    print("No qualifying median image found.")
else:
    out_png = OUT_DIR / f"median_two_panel_{MODEL_KEY}_{Path(median_path).stem}.png"
    visualize_two_panel(
        img_path=median_path,
        model=model,
        best_prompt=best_prompt,
        baseline_prompt=BASELINE_PROMPT,
        best_conf=float(best_conf),
        lbl_dir=str(LBL_DIR),
        conf_sweep=CONF_SWEEP,
        iou_threshold=IOU_THRESHOLD,
        output_path=out_png,
        model_key=MODEL_KEY,
        model_label=MODEL_KEY,
        scored_images=scored,
        min_gt_boxes=MIN_GT_BOXES,
    )
    print(f"Saved: {out_png}")